In [1]:
import pandas as pd
import numpy as np

In [2]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [4]:
query = """
SELECT
    -- old station native_id (from DB)
    (SELECT s.native_id
     FROM meta_station s
     WHERE s.station_id = %s
    ) AS old_native_id_db,

    -- old count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_old,

    -- new count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s
     ) t
    ) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
    FROM meta_history m_old
    JOIN obs_raw o_old
    ON m_old.history_id = o_old.history_id

    JOIN meta_history m_new
    ON m_new.station_id = %s
    JOIN obs_raw o_new
    ON m_new.history_id = o_new.history_id
    AND o_old.obs_time = o_new.obs_time
    AND o_old.vars_id  = o_new.vars_id

    WHERE m_old.station_id = %s
    AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum
"""
def count_station_overlap(old_station_id, new_station_id, engine):

    params = (
        old_station_id,  # old_native_id_db
        old_station_id,  # n_old
        new_station_id,  # n_new
        old_station_id,  # n_overlap (old)
        new_station_id,  # n_overlap (new)
        new_station_id,  # n_overlap_same_datum (new)
        old_station_id,  # n_overlap_same_datum (old)
    )

    df = pd.read_sql(query, engine, params=params)
    return df.iloc[0]


stats = count_station_overlap(
    old_station_id=13185,
    new_station_id=13256,
    engine=engine
)

stats

old_native_id_db        McDonn
n_old                   124252
n_new                   353725
n_overlap               124104
n_overlap_same_datum    124104
Name: 0, dtype: object